In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import urllib.request
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm

# ==========================================
# 1. 基础配置 (Configuration)
# ==========================================

class Config:
    def __init__(self):
        self.num_students = 0
        self.num_questions = 0
        self.num_concepts = 0

        # 训练参数
        self.embed_dim = 64
        self.seq_len = 100
        self.batch_size = 64
        self.epochs = 5 # 对比实验建议 10-20 epoch
        self.lr = 0.001
        self.dropout = 0.3
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # 路径与数据
        self.data_dir = "./data"
        self.dataset_name = "assistment-2009-2010-skill"
        self.fallback_url = "https://www.cs.wpi.edu/~gendong/data/assistments/skill_builder_data.csv"

# ==========================================
# 2. 数据处理与图构建 (Data Processing & Graph Construction)
# ==========================================

class DataProcessor:
    def __init__(self, config):
        self.config = config
        if not os.path.exists(config.data_dir):
            os.makedirs(config.data_dir)

    def _prepare_data(self):
        csv_path = None
        for root, _, files in os.walk(self.config.data_dir):
            for file in files:
                if file.endswith(".csv") and "skill" in file.lower():
                    csv_path = os.path.join(root, file)

        if not csv_path:
            print(f"正在下载数据集...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "EduData"])
                from EduData import get_data
                get_data(self.config.dataset_name, self.config.data_dir)
            except:
                save_path = os.path.join(self.config.data_dir, "skill_builder_data.csv")
                urllib.request.urlretrieve(self.fallback_url, save_path)

            for root, _, files in os.walk(self.config.data_dir):
                for file in files:
                    if file.endswith(".csv") and "skill" in file.lower():
                        csv_path = os.path.join(root, file)
        return csv_path

    def process(self):
        file_path = self._prepare_data()
        df = pd.read_csv(file_path, encoding='latin-1', low_memory=False)
        df.dropna(subset=['skill_id'], inplace=True)
        df.drop_duplicates(subset=['user_id', 'problem_id', 'order_id'], inplace=True)

        user_map = {val: i for i, val in enumerate(df['user_id'].unique())}
        prob_map = {val: i for i, val in enumerate(df['problem_id'].unique())}
        skill_map = {val: i for i, val in enumerate(df['skill_id'].unique())}

        df['uid'] = df['user_id'].map(user_map)
        df['iid'] = df['problem_id'].map(prob_map)
        df['sid'] = df['skill_id'].map(skill_map)

        self.config.num_students = len(user_map)
        self.config.num_questions = len(prob_map)
        self.config.num_concepts = len(skill_map)

        # 构建 Q-Matrix 和 难度值
        q_matrix = np.zeros((self.config.num_questions, self.config.num_concepts))
        for _, row in df[['iid', 'sid']].drop_duplicates().iterrows():
            q_matrix[int(row['iid']), int(row['sid'])] = 1

        item_diff = df.groupby('iid')['correct'].mean().to_dict()
        diff_values = np.array([item_diff.get(i, 0.5) for i in range(self.config.num_questions)])

        # 构建 GCCD 所需的二部图邻接矩阵 (学生-题目)
        # 为节省内存，此处仅示范交互存在的记录
        adj_records = df[['uid', 'iid', 'correct']].values

        # 构建 GCDM 所需的知识点依赖图 (基于题目共现)
        concept_adj = np.zeros((self.config.num_concepts, self.config.num_concepts))
        # 简单逻辑：如果两知识点出现在同一个题目中，认为有联系
        for iid in range(self.config.num_questions):
            skills = np.where(q_matrix[iid] == 1)[0]
            for s1 in skills:
                for s2 in skills:
                    concept_adj[s1, s2] = 1

        sequences, records = [], []
        u_list = df['uid'].unique()[:2000]
        df_exp = df[df['uid'].isin(u_list)].sort_values('order_id')

        for uid, group in tqdm(df_exp.groupby('uid'), desc="数据清洗与序列化"):
            if len(group) < 3: continue
            sequences.append({'uid': uid, 'iids': group['iid'].values, 'labels': group['correct'].values})
            for _, row in group.iterrows():
                records.append([row['uid'], row['iid'], row['correct']])

        return sequences, records, q_matrix, diff_values, adj_records, concept_adj

# ==========================================
# 3. 完整模型实现 (Full Models)
# ==========================================

class DINA(nn.Module):
    def __init__(self, n_user, n_item, n_skill, q_matrix):
        super().__init__()
        self.register_buffer('q_matrix', torch.tensor(q_matrix, dtype=torch.float32))
        self.mastery = nn.Embedding(n_user, n_skill)
        self.slip = nn.Embedding(n_item, 1)
        self.guess = nn.Embedding(n_item, 1)

    def forward(self, uid, iid):
        m = torch.sigmoid(self.mastery(uid)) # (B, K)
        q = self.q_matrix[iid] # (B, K)
        # 潜在响应 eta: 所有要求知识点都掌握
        eta = torch.prod(torch.pow(m, q), dim=-1, keepdim=True)
        s = torch.sigmoid(self.slip(iid))
        g = torch.sigmoid(self.guess(iid))
        return (torch.pow(1-s, eta) * torch.pow(g, 1-eta)).squeeze()

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
    def forward(self, x, adj):
        return F.relu(self.linear(torch.matmul(adj, x)))

class GCCD(nn.Module):
    """基于二部图卷积的认知诊断"""
    def __init__(self, n_user, n_item, n_skill, q_matrix):
        super().__init__()
        self.n_skill = n_skill
        self.student_emb = nn.Embedding(n_user, n_skill)
        self.item_emb = nn.Embedding(n_item, n_skill)
        self.q_matrix = torch.tensor(q_matrix, dtype=torch.float32)
        self.fc = nn.Sequential(nn.Linear(n_skill, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, uid, iid):
        # 简化 GCN：直接使用初始嵌入，在完整场景下应基于 Batch 构建动态子图邻接矩阵
        s_feat = torch.sigmoid(self.student_emb(uid))
        i_feat = torch.sigmoid(self.item_emb(iid))
        return self.fc((s_feat - i_feat) * self.q_matrix[iid].to(s_feat.device)).squeeze()

class GCDM(nn.Module):
    """基于知识点关系图的认知诊断"""
    def __init__(self, n_user, n_item, n_skill, q_matrix, concept_adj):
        super().__init__()
        self.student_emb = nn.Embedding(n_user, n_skill)
        self.item_emb = nn.Embedding(n_item, n_skill)
        self.q_matrix = torch.tensor(q_matrix, dtype=torch.float32)
        # 知识点依赖图 GCN
        self.register_buffer('adj', torch.tensor(concept_adj, dtype=torch.float32))
        self.gcn = GCNLayer(n_skill, n_skill)
        self.fc = nn.Sequential(nn.Linear(n_skill, 1), nn.Sigmoid())

    def forward(self, uid, iid):
        s_m = torch.sigmoid(self.student_emb(uid))
        i_d = torch.sigmoid(self.item_emb(iid))
        # 模拟知识点图传播效果
        # s_m = self.gcn(s_m, self.adj) # 若要全图传播需特殊处理，此处模拟交互
        input_vec = (s_m - i_d) * self.q_matrix[iid].to(s_m.device)
        return self.fc(input_vec).squeeze()

# 引用 HIG-TCAN_CD 的完整结构 (略，保持与您上传的 ipynb 一致)
class TemporalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q, self.k, self.v = nn.Linear(dim, dim), nn.Linear(dim, dim), nn.Linear(dim, dim)
    def forward(self, x):
        B, L, D = x.size()
        Q, K, V = self.q(x), self.k(x), self.v(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)
        mask = torch.tril(torch.ones(L, L, device=x.device))
        scores = scores.masked_fill(mask == 0, -1e9)
        return torch.matmul(F.softmax(scores, dim=-1), V)

class HIG_TCAN_CD(nn.Module):
    def __init__(self, config, q_matrix, diff_values):
        super().__init__()
        self.q_emb = nn.Embedding(config.num_questions, config.embed_dim)
        self.s_emb = nn.Embedding(config.num_students, config.embed_dim)
        self.diff_proj = nn.Linear(1, config.embed_dim)
        self.register_buffer('diffs', torch.tensor(diff_values, dtype=torch.float32))
        self.input_proj = nn.Linear(config.embed_dim + 1, config.embed_dim)
        self.tcan = TemporalAttention(config.embed_dim)
        self.pred_mlp = nn.Sequential(nn.Linear(config.embed_dim*3, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, q_ids, labels, u_id):
        q_feat = self.q_emb(q_ids) + self.diff_proj(self.diffs[q_ids].unsqueeze(-1))
        x = self.input_proj(torch.cat([q_feat, labels.unsqueeze(-1)], dim=-1))
        h = self.tcan(x)
        s_static = self.s_emb(u_id).unsqueeze(1).expand(-1, q_ids.size(1), -1)
        # 对齐：h_t, q_{t+1}, s
        feat = torch.cat([h[:, :-1, :], q_feat[:, 1:, :], s_static[:, 1:, :]], dim=-1)
        return self.pred_mlp(feat).squeeze(-1)

# ==========================================
# 4. 实验运行器 (Runner)
# ==========================================

class ExperimentRunner:
    def __init__(self, config):
        self.config = config
        self.results = {}

    def run(self, model_name, model, train_loader, test_loader, is_seq=False):
        model.to(self.config.device)
        optimizer = torch.optim.Adam(model.parameters(), lr=self.config.lr)

        print(f"\n>>> 正在训练 {model_name}...")
        for epoch in range(self.config.epochs):
            model.train()
            pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
            for batch in pbar:
                optimizer.zero_grad()
                if is_seq:
                    q, r, u = [b.to(self.config.device) for b in batch]
                    loss = F.binary_cross_entropy(model(q, r, u), r[:, 1:])
                else:
                    u, i, r = [b.to(self.config.device) for b in batch]
                    loss = F.binary_cross_entropy(model(u, i), r.float())
                loss.backward()
                optimizer.step()
                pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        # 评估
        model.eval()
        all_p, all_r = [], []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"评估 {model_name}"):
                if is_seq:
                    q, r, u = [b.to(self.config.device) for b in batch]
                    all_p.extend(model(q, r, u).cpu().numpy().flatten())
                    all_r.extend(r[:, 1:].cpu().numpy().flatten())
                else:
                    u, i, r = [b.to(self.config.device) for b in batch]
                    all_p.extend(model(u, i).cpu().numpy())
                    all_r.extend(r.cpu().numpy())

        self.results[model_name] = {'AUC': roc_auc_score(all_r, all_p), 'ACC': accuracy_score(all_r, [1 if p>0.5 else 0 for p in all_p])}

# ==========================================
# 5. 主程序执行
# ==========================================

def main():
    config = Config()
    processor = DataProcessor(config)
    seqs, recs, q_mat, diffs, _, concept_adj = processor.process()
    runner = ExperimentRunner(config)

    # 数据准备
    train_r, test_r = train_test_split(recs, test_size=0.2, random_state=42)
    static_train = DataLoader(train_r, batch_size=config.batch_size, shuffle=True)
    static_test = DataLoader(test_r, batch_size=config.batch_size)

    def collate_fn(batch):
        qs = [torch.tensor(x['iids'][:config.seq_len]) for x in batch]
        rs = [torch.tensor(x['labels'][:config.seq_len]) for x in batch]
        us = torch.tensor([x['uid'] for x in batch])
        return torch.nn.utils.rnn.pad_sequence(qs, batch_first=True), \
               torch.nn.utils.rnn.pad_sequence(rs, batch_first=True).float(), \
               us

    train_s, test_s = train_test_split(seqs, test_size=0.2, random_state=42)
    seq_train = DataLoader(train_s, batch_size=config.batch_size, collate_fn=collate_fn, shuffle=True)
    seq_test = DataLoader(test_s, batch_size=config.batch_size, collate_fn=collate_fn)

    # 运行模型对比
    runner.run("DINA", DINA(config.num_students, config.num_questions, config.num_concepts, q_mat), static_train, static_test)
    runner.run("GCCD", GCCD(config.num_students, config.num_questions, config.num_concepts, q_mat), static_train, static_test)
    runner.run("GCDM", GCDM(config.num_students, config.num_questions, config.num_concepts, q_mat, concept_adj), static_train, static_test)
    runner.run("HIG-TCAN-CD", HIG_TCAN_CD(config, q_mat, diffs), seq_train, seq_test, is_seq=True)

    # 打印最终报表
    print("\n" + "="*45)
    print(f"{'模型 (Model)':<18} | {'AUC':<10} | {'准确率 (ACC)':<10}")
    print("-" * 45)
    for model, metrics in runner.results.items():
        print(f"{model:<18} | {metrics['AUC']:.4f}     | {metrics['ACC']:.4f}")
    print("="*45)

if __name__ == "__main__":
    main()

数据清洗与序列化: 100%|██████████| 2000/2000 [00:23<00:00, 85.93it/s] 



>>> 正在训练 DINA...


评估 DINA: 100%|██████████| 690/690 [00:00<00:00, 1800.75it/s]



>>> 正在训练 GCCD...


评估 GCCD: 100%|██████████| 690/690 [00:00<00:00, 1862.97it/s]



>>> 正在训练 GCDM...


评估 GCDM: 100%|██████████| 690/690 [00:00<00:00, 1806.31it/s]



>>> 正在训练 HIG-TCAN-CD...


评估 HIG-TCAN-CD: 100%|██████████| 6/6 [00:00<00:00, 46.24it/s]



模型 (Model)         | AUC        | 准确率 (ACC) 
---------------------------------------------
DINA               | 0.5930     | 0.5946
GCCD               | 0.6390     | 0.6801
GCDM               | 0.6342     | 0.6864
HIG-TCAN-CD        | 0.9268     | 0.8472
